In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


customers_url = "D:/Python Projects/day1/data/raw/raw_customers.csv"
order_items_url = "D:/Python Projects/day1/data/raw/raw_order_items.csv"
orders_url = "D:/Python Projects/day1/data/raw/raw_orders.csv"
products_url = "D:/Python Projects/day1/data/raw/raw_products.csv"


customers = pd.read_csv(customers_url)
order_items = pd.read_csv(order_items_url)
orders = pd.read_csv(orders_url)
products = pd.read_csv(products_url)



In [10]:
# creating validation report 

validation_report = {}

validation_report["duplicated_customers"] = customers["customer_id"].duplicated().sum()

validation_report["missing_cities"] = customers["city"].isna().sum()

validation_report["missing_categories"] = products["category"].isna().sum()

validation_report["illogical_cost"] = (products["unit_cost"] > products["unit_price"]).sum()

validation_report["Invalid order dates"] = pd.to_datetime(orders["order_date"], errors="coerce").isna().sum()

validation_report["Orders with missing customer IDs"] = orders["customer_id"].isna().sum()

validation_report["Negative or zero quantity"] = (order_items["quantity"] <= 0).sum()

validation_report["Items with missing product IDs"] = order_items["product_id"].isna().sum()

print("Validation Report")
print("-----------------")

for check_name, result in validation_report.items():
    print(f"{check_name}: {result}")


Validation Report
-----------------
duplicated_customers: 1
missing_cities: 6
missing_categories: 1
illogical_cost: 1
Invalid order dates: 1
Orders with missing customer IDs: 0
Negative or zero quantity: 5
Items with missing product IDs: 0


In [11]:
# fixing the errors in data

# customer data cleanning 
customers_clean = customers.copy()
customers_clean = customers.drop_duplicates(subset=["customer_id"], keep="first")
customers_clean["city"] = customers["city"].fillna("Unknown")

# products data cleanning 
products_clean = products.copy()
products_clean = products.dropna(subset="product_id")
products_clean["category"] = products["category"].fillna("Uncategorized")

# orders data cleanning 
orders_clean = orders.copy()
orders_clean = orders.dropna(subset="order_date")
orders_clean = orders.drop_duplicates(subset="order_id", keep="first")
orders_clean["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")


# customers data cleanning 
customers_clean = customers.copy()
customers_clean = customers.dropna(subset="customer_id")

# order_items data cleanning 
order_items_clean = order_items.copy()
order_items_clean = order_items[order_items["quantity"] > 0]

# saving the cleaned files to data/clean path 

output_path = Path("D:\Python Projects\day1\data\clean")
output_path.mkdir(parents=True, exist_ok=True)

customers_clean.to_csv(output_path / "clean_customers.csv", index = False)
products_clean.to_csv(output_path / "clean_products.csv", index=False)
orders_clean.to_csv(output_path / "orders_clean.csv", index=False)
order_items_clean.to_csv(output_path / "order_items_clean.csv", index=False)


<>:30: SyntaxWarning: invalid escape sequence '\P'
<>:30: SyntaxWarning: invalid escape sequence '\P'
C:\Users\moham\AppData\Local\Temp\ipykernel_13064\1116769753.py:30: SyntaxWarning: invalid escape sequence '\P'
  output_path = Path("D:\Python Projects\day1\data\clean")


In [12]:
# creating fct table 

# merge with customers 
sales_fact = orders_clean.merge(
    customers_clean,
    on="customer_id",
    how="inner"
)

# merge with order_items  
sales_fact = sales_fact.merge(
    order_items_clean,
    on="order_id",
    how="inner"
)

# merge with products 

sales_fact = sales_fact.merge(
    products_clean,
    on="product_id",
    how="inner"
)

sales_fact["month"] = sales_fact["order_date"].dt.to_period("M").dt.to_timestamp()
sales_fact["gross_sales"] = sales_fact["quantity"] * sales_fact["unit_price_at_order"]
sales_fact["discount_amount"] = sales_fact["gross_sales"] * sales_fact["discount_pct"] / 100
sales_fact["net_sales"] = sales_fact["gross_sales"] - sales_fact["discount_amount"]
sales_fact["cost_amount"] = sales_fact["quantity"] * sales_fact["unit_cost"]
sales_fact["profit"] = sales_fact["net_sales"] - sales_fact["cost_amount"]

In [13]:
# creating data marts 

# monthly KPIs


monthly_kpis = sales_fact.groupby("month").agg(
    total_orders=("order_id", "count"),
    total_customers=("customer_id", "nunique"),
    gross_sales=("gross_sales", "sum"),
    net_sales=("net_sales", "sum"),
    profit=("profit", "sum"),
    avg_order_value=("net_sales", "mean")
)

# product_summary
product_summary = sales_fact.groupby(["product_id", "product_name", "category"]).agg(
    net_sales=("net_sales", "sum"),
    profit=("profit", "sum")
)

# customer summary 
customer_summary = sales_fact.groupby(["customer_id", "customer_name", "city", "segment"]).agg(
    total_orders=("order_id", "count"),
    net_sales=("net_sales", "sum"),
    first_order_date=("order_date", "min"),
    last_order_date=("order_date", "max")
)

agg_path = Path("D:\Python Projects\day1\data\marts")

monthly_kpis.to_csv(agg_path / "monthly_kpis.csv", index=False)
product_summary.to_csv(agg_path / "product_summary.csv", index=False)
customer_summary.to_csv(agg_path / "customer_summary.csv", index=False)

<>:29: SyntaxWarning: invalid escape sequence '\P'
<>:29: SyntaxWarning: invalid escape sequence '\P'
C:\Users\moham\AppData\Local\Temp\ipykernel_13064\4134748929.py:29: SyntaxWarning: invalid escape sequence '\P'
  agg_path = Path("D:\Python Projects\day1\data\marts")
